In [1]:
import numpy as np
import pandas as pd
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import warnings
warnings.simplefilter("ignore", UserWarning)

nlp = spacy.load("en_core_web_lg")

# ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
# ruler.from_disk("entity_rulers.jsonl")

# patterns = [
# {"label":"CALLSIGN","pattern":[{"LOWER":{"IN":["delta","southwest","united","american","jetblue","eagle","alaska","frontier","ups","airfrans"]}},{"TEXT":{"REGEX":"^[0-9]{1,4}$"}}]},
# {"label":"CALLSIGN","pattern":[{"LOWER":"air"},{"LOWER":"canada"},{"TEXT":{"REGEX":"^[0-9]{1,4}$"}}]},
# {"label":"CALLSIGN","pattern":[{"LOWER":"speed"},{"LOWER":"bird"},{"TEXT":{"REGEX":"^[0-9]{1,4}$"}}]},
# {"label":"CALLSIGN","pattern":[{"LOWER":"japan"},{"LOWER":"air"},{"TEXT":{"REGEX":"^[0-9]{1,4}$"}}]},

# {"label":"ACSTATE","pattern":[{"LOWER":{"IN":["hold","taxi","go","approach"]}}]},
# {"label":"ACSTATE","pattern":[{"LOWER":"turn"},{"LOWER":{"IN":["left","right"]}}]},

# {"label":"DESTINATION","pattern":[{"TEXT":{"REGEX":"^K[A-Z]{3}$"}}]},
# {"label":"DESTINATION","pattern":[{"LOWER":"taxiway"},{"LOWER":"alpha"}]},
# {"label":"DESTINATION","pattern":[{"LOWER":"taxiway"},{"LOWER":"bravo"}]},
# {"label":"DESTINATION","pattern":[{"LOWER":"taxiway"},{"LOWER":"charlie"}]},
# {"label":"DESTINATION","pattern":[{"LOWER":"taxiway"},{"LOWER":"mike"}]},
# {"label":"DESTINATION","pattern":[{"LOWER":"holding"},{"LOWER":"point"},{"TEXT":{"REGEX":"^[A-Za-z][0-9]{1,2}$"}}]}
# ]

# ruler.add_patterns(patterns)


/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
text = ["17:43:26 Delta 276, Tokyo Tower, good evening, taxi to holding point C1."]
for txt in text:
    print(f"text: {txt}")
    doc = nlp(txt)
    for ent in doc.ents:
        print(ent.text, ent.label_)

text: 17:43:26 Delta 276, Tokyo Tower, good evening, taxi to holding point C1.
17:43:26 CARDINAL
Delta 276 ORG
Tokyo Tower FAC
evening TIME
point C1 PRODUCT


In [3]:
! python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
! python -m spacy train config.cfg --output ./tok2vec --paths.train ./train_data.spacy --paths.dev ./val_data.spacy

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: tok2vec
ℹ Using CPU

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.0

In [ ]:
from spacy.training import Corpus

nlp_ner = spacy.load("./tok2vec/model-best")

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']

=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.854
Recall:    0.751
F1-Score:  0.799


In [42]:
import glob
test_file = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
file_idx = 1

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [ ]:
nlp_ner = spacy.load("./tok2vec/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [49]:
nlp_ner = spacy.load("./tok2vec/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")


# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']


=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.755
Recall:    0.822
F1-Score:  0.787
